In [1]:
import sys
from tqdm import tqdm
import yaml
import os
from typing import List, Tuple, Dict, Generator
import json
import numpy as np
import joblib
import sys
import json
from tqdm import tqdm
from typing import Dict
import numpy as np
import torch
import os
from time import time
import gc

import nltk
nltk.download('punkt_tab')
nltk.download('wordnet')

# TO CHANGE
BASEDIR = "../../"
sys.path.insert(0, BASEDIR)

[nltk_data] Downloading package punkt_tab to /home/dzigen/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /home/dzigen/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
from src.kg_model import KnowledgeGraphModel
from src.pipelines.qa import QAPipelineConfig, QAPipeline


from src.pipelines.qa.knowledge_retriever import AStarGraphSearchConfig, AStarMetricsConfig, BFSSearchConfig, MixturedGraphSearchConfig
from src.db_drivers.kv_driver import KVDBConnectionConfig, KeyValueDriverConfig
from src.db_drivers.kv_driver.connectors import DEFAULT_MIXEDKV_CONFIG
from src.pipelines.qa.knowledge_retriever.TripletsFilter import TripletsFilterConfig
from src.pipelines.qa import QueryLLMParserConfig, KnowledgeComparatorConfig, KnowledgeRetrieverConfig, QALLMGeneratorConfig


from src.utils import NodeType, Logger

/home/dzigen/Desktop/PersonalAI/.pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Create configs

In [3]:
# retrieve
RETRIVER_CONFIG_DUMP = "retriever_config" 

DEFAULT_MIXEDKV_CONFIG.params['redis_config'].host = 'localhost'
DEFAULT_MIXEDKV_CONFIG.params['mongo_config'].host = 'localhost'

KV_STORAGE_CONFIG = KeyValueDriverConfig(db_vendor='mixed_kv', db_config=DEFAULT_MIXEDKV_CONFIG)

astar_config= AStarGraphSearchConfig(
    metrics_config=AStarMetricsConfig(h_metric_name='ip', kvdriver_config=KV_STORAGE_CONFIG),
    max_depth=8, max_passed_nodes=500,
    accepted_node_types=[NodeType.object , NodeType.hyper, NodeType.episodic]
)

bfs_config = BFSSearchConfig(
    strict_filter=True,
    hyper_episodic_num=25,
    chain_triplets_num=25,
    other_triplets_num=6
)

retriever_config = MixturedGraphSearchConfig(astar_config=astar_config, bfs_config=bfs_config)

joblib.dump(retriever_config, RETRIVER_CONFIG_DUMP)

# filter config
FILTER_CONFIG_DUMP = "filter_config"

filter_config = TripletsFilterConfig(
    max_k=50
)

joblib.dump(filter_config, FILTER_CONFIG_DUMP)

['filter_config']

## Loading hyperparameters

In [3]:
# Read YAML file
with open("params.yaml", 'r') as stream:
    HYPER_PARAMS = yaml.safe_load(stream)

BASE_PATH = "../../data/knowledge_graphs/"
DATASET_PATH = BASE_PATH + f"{HYPER_PARAMS['dataset_name']}/"
KG_PATH = DATASET_PATH + f"{HYPER_PARAMS['kg_name']}/"

GRAPH_DRIVER_CONFIG_PATH = KG_PATH + "graph_config"
EMBEDDINGS_DRIVER_CONFIG_PATH = KG_PATH + "embeddings_config"

EXPERIMENT_DIR = f"{HYPER_PARAMS['dataset_name']}/exp_logs/{HYPER_PARAMS['experiment_name']}"
GENERATED_ANSWERS_DIR = f'{EXPERIMENT_DIR}/answer_packs'
METRICS_DIR = f'{EXPERIMENT_DIR}/metric_packs'

TMP_GENERATED_ANSWERS_DIR = f'{EXPERIMENT_DIR}/tmp_answer_packs'

QA_ELAPSED_TIME = f'{EXPERIMENT_DIR}/elapsed_time.json'

HYPERPARAMS_SAVE_PATH = f'{EXPERIMENT_DIR}/hyperparams.json'
QA_CONFIG_SAVE_PATH = f'{EXPERIMENT_DIR}/qa_config'
RETRIEVER_CONFIG_SAVE_PATH = f'{EXPERIMENT_DIR}/retirver_config'
FILTER_CONFIG_SAVE_PATH = f'{EXPERIMENT_DIR}/filter_config'

In [4]:
# инициализируем граф знаний

graph_config = joblib.load(GRAPH_DRIVER_CONFIG_PATH)
embed_config = joblib.load(EMBEDDINGS_DRIVER_CONFIG_PATH)

# !!! IMPORTANT !!!
graph_config.driver_config.db_config.need_to_clear = False
embed_config.nodesdb_driver_config.db_config.need_to_clear = False
embed_config.tripletsdb_driver_config.db_config.need_to_clear = False
# !!! IMPORTANT !!!

# fixing paths to vector dbs
embed_config.embedder_config.model_name_or_path = '/'.join(embed_config.embedder_config.model_name_or_path.split("/")[1:])
embed_config.nodesdb_driver_config.db_config.path = '/'.join(embed_config.nodesdb_driver_config.db_config.path.split("/")[1:])
embed_config.tripletsdb_driver_config.db_config.path = '/'.join(embed_config.tripletsdb_driver_config.db_config.path.split("/")[1:])

kg_model = KnowledgeGraphModel(
    graph_config=graph_config,
    embeddings_config=embed_config)

print(kg_model.embeddings_struct.vectordbs['nodes'].count_items())
print(kg_model.embeddings_struct.vectordbs['triplets'].count_items())
print(kg_model.graph_struct.db_conn.count_items())

No sentence-transformers model found with name ../../models/intfloat/multilingual-e5-small. Creating a new one with mean pooling.


49597
44328
{'triplets': 182838, 'nodes': 49597}


In [5]:
# задаём конфигурацию qa-пайплайна

retriever_config = joblib.load(HYPER_PARAMS['knowledge_retriever']['retriever_config_path'])
filter_config = joblib.load(HYPER_PARAMS['knowledge_retriever']['filter_config_path'])

qa_config = QAPipelineConfig(
    query_parser_config=QueryLLMParserConfig(lang=HYPER_PARAMS['language']),

    knowledge_comparator_config=KnowledgeComparatorConfig(),
    
    knowledge_retriever_config=KnowledgeRetrieverConfig(
        retriever_method=HYPER_PARAMS['knowledge_retriever']['retriever_method'], retriever_config=retriever_config,
        filter_method=HYPER_PARAMS['knowledge_retriever']['filter_method'], filter_config=filter_config),
    
    answer_generator_config=QALLMGeneratorConfig(lang=HYPER_PARAMS['language']))

In [6]:
qa_pipeline = QAPipeline(kg_model, qa_config)

## Structure init

In [7]:
if not os.path.exists(HYPER_PARAMS['dataset_name']):
    raise ValueError("Директории не существует")

if os.path.exists(EXPERIMENT_DIR):
    raise ValueError("Директория существует")

if os.path.exists(GENERATED_ANSWERS_DIR):
    raise ValueError("Директория существует")

if os.path.exists(METRICS_DIR):
    raise ValueError("Директория существует")

# создать каталог
os.mkdir(EXPERIMENT_DIR)
# создать каталог для ответов
os.mkdir(GENERATED_ANSWERS_DIR)
# создать каталог для метрик
os.mkdir(METRICS_DIR)

os.mkdir(TMP_GENERATED_ANSWERS_DIR)

# сохранить параметры
with open(HYPERPARAMS_SAVE_PATH, 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(HYPER_PARAMS, indent=1, ensure_ascii=False))

# сохранить конфиги
joblib.dump(qa_config, QA_CONFIG_SAVE_PATH)
joblib.dump(retriever_config, RETRIEVER_CONFIG_SAVE_PATH)
joblib.dump(filter_config, FILTER_CONFIG_SAVE_PATH)

['diaasqa/exp_logs/bfs_with_gigachat_full/filter_config']

## Loading questions

In [4]:
def diaasqa_loading(dir_path: str):
    pack_files = os.listdir(dir_path)
    packs = []

    for pack_f in pack_files:
        with open(f"{dir_path}/{pack_f}", 'r', encoding='utf-8') as fd:
            data = json.loads(fd.read())

        pack_name = '.'.join(pack_f.split('.')[:-1])
        questions = list(map(lambda item: item['question'], data)) 
        answers = list(map(lambda item: item['answer'], data))

        packs.append((pack_name, questions, answers))

    return packs

In [5]:
DATASET_LOADERS = {
    'diaasqa': diaasqa_loading
}

## Starting qa process

In [10]:
question_packs = DATASET_LOADERS[HYPER_PARAMS['dataset_name']](HYPER_PARAMS['eval_dataset_path'])

In [19]:
for pack_name, questions, _ in question_packs[11:]:
    
    pack_tmp_dir = f"{TMP_GENERATED_ANSWERS_DIR}/{pack_name}"
    if not os.path.exists(pack_tmp_dir):
        os.mkdir(pack_tmp_dir)

    process = tqdm(range(len(questions)))
    for i in process:
        process.set_postfix_str(pack_name)

        s_time = time()
        answer, info = qa_pipeline.answer(questions[i])
        e_time = time()

        answer_dump_file = f"{pack_tmp_dir}/answer_{i}"
        joblib.dump({'answer': answer, 'info': info, 'elapsed_time': e_time - s_time}, answer_dump_file)

100%|██████████| 200/200 [10:10<00:00,  3.05s/it, device_sentiment]


In [22]:
# accumulate generate answers
elapsed_times = {}

for pack_name, _, _ in question_packs:

    pack_tmp_dir = f"{TMP_GENERATED_ANSWERS_DIR}/{pack_name}"

    if not os.path.exists(pack_tmp_dir):
        print("Папки с ответами не сущестует: ", pack_name)
        continue

    tmp_answer_dumps = os.listdir(pack_tmp_dir)

    accum_answers = dict()
    elapsed_times[pack_name] = {'per_question': []}
    for tmp_dump in tqdm(tmp_answer_dumps):
        answer_info = joblib.load(f"{pack_tmp_dir}/{tmp_dump}")
        answer_num = int(tmp_dump.split("_")[1])
        accum_answers[answer_num] = answer_info['answer']
        elapsed_times[pack_name]['per_question'].append(answer_info['elapsed_time'])

    elapsed_times[pack_name]['sum'] = sum(elapsed_times[pack_name]['per_question'])
    elapsed_times[pack_name]['mean'] = np.mean(elapsed_times[pack_name]['per_question'])
    elapsed_times[pack_name]['median'] = np.median(elapsed_times[pack_name]['per_question'])
    
    answers_pack_path = f"{GENERATED_ANSWERS_DIR}/{pack_name}.json"
    with open(answers_pack_path, 'w', encoding='utf-8') as fd:
        fd.write(json.dumps(accum_answers, indent=1, ensure_ascii=False))

with open(QA_ELAPSED_TIME, 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(elapsed_times, indent=1, ensure_ascii=False))

100%|██████████| 200/200 [00:00<00:00, 14664.89it/s]


## Measuring answers quality

In [6]:
# Source: https://amitness.com/2020/08/information-retrieval-evaluation/

#Retrieval metrics
# - mAP
# - MRR
# - precision
# - recall
# - f1
#Reader metrics
# - BLEU presision
# - ROUGE recall
# - METEOR f1

from torchmetrics.text.rouge import ROUGEScore
from torchmetrics.text import BLEUScore
from evaluate import load
import evaluate
import numpy as np
from typing import List
from tqdm import tqdm
from torchmetrics.text.bert import BERTScore
from Levenshtein import distance as levenshtain_distance

class ReaderMetrics:
    def __init__(self, model_path, base_dir = '../..'):
        self.rouge_obj = ROUGEScore()
        self.bleu1_obj = BLEUScore(n_gram=1)
        self.bleu2_obj = BLEUScore(n_gram=2)
        print("Loading Meteor...")
        self.meteor_obj = evaluate.load("./metrics/meteor")
        print("Loading ExactMatch")
        self.em_obj = evaluate.load("./metrics/exact_match")
        self.bertscore_obj = BERTScore(f"{base_dir}/models/{model_path}", return_hash=True)
        
    def bertscore(self, predicted: List[str], targets: List[str]):
        output = self.bertscore_obj(predicted, targets)
        output['precision'] = round(float(output['precision'].mean()), 5)
        output['recall'] = round(float(output['recall'].mean()), 5)
        output['f1'] = round(float(output['f1'].mean()), 5)
        
        return output
    
    def rougel(self, predicted: List[str], targets: List[str]):
        return [self.rouge_obj(
            predicted[i], targets[i])['rougeL_fmeasure'] 
                 for i in range(len(targets))]
        
    def bleu1(self, predicted: List[str], targets: List[str]):
        return [self.bleu1_obj(
            [predicted[i]], [[targets[i]]])
                 for i in range(len(targets))]
        
    def bleu2(self, predicted: List[str], targets: List[str]):
        return [self.bleu2_obj(
            [predicted[i]], [[targets[i]]]) 
                 for i in range(len(targets))]
        
    def meteor(self, predicted: List[str], targets: List[str]):
        return [self.meteor_obj.compute(
            predictions=[predicted[i]], references=[targets[i]])['meteor'] 
                 for i in range(len(targets))]
        
    def exact_match(self, predicted: List[str], targets: List[str]):
        return [self.em_obj.compute(
            predictions=[predicted[i]], references=[targets[i]], ignore_case=True, ignore_punctuation=True)["exact_match"]
                for i in range(len(targets))]
    
    def levenshtain_score(self, predicted: List[str], targets: List[str]):
        return list(map(lambda pair: levenshtain_distance(pair[1], pair[0]), zip(predicted, targets)))

In [7]:
def loading_generated_pack(base_dir: str, pack_name) -> Dict[int,str]:
    with open(f"{base_dir}/{pack_name}.json", 'r', encoding='utf-8') as fd:
        data = json.loads(fd.read())
    return data

def round5(number: float) -> float:
    return round(number, 5)

def save_json(data: Dict[str, object], save_path: str):
    dump = json.dumps(data, ensure_ascii=False, indent=1)
    with open(f"{save_path}.json", 'w', encoding='utf-8') as fd:
        fd.write(dump)

In [8]:
BERTSCORE_MODEL_PATH = "google/electra-base-discriminator"
METRICS = ReaderMetrics(base_dir="../..", model_path=BERTSCORE_MODEL_PATH)

Loading Meteor...
Loading ExactMatch


In [9]:
# загружаем датасте 
question_packs = DATASET_LOADERS[HYPER_PARAMS['dataset_name']](HYPER_PARAMS['eval_dataset_path'])

In [14]:
for pack_name, _, all_target_answers in tqdm(question_packs):

    torch.cuda.empty_cache()
    gc.collect()

    generated_pack = loading_generated_pack(GENERATED_ANSWERS_DIR, pack_name)
    
    filtered_target_answers = []
    generated_answers = []
    none_answers = 0
    for i, answer in generated_pack.items():
        if answer is not None:
            generated_answers.append(answer)
            filtered_target_answers.append(all_target_answers[int(i)])
        else:
            none_answers += 1

    b1_scores = round5(np.mean(METRICS.bleu1(generated_answers, filtered_target_answers)))
    b2_scores  = round5(np.mean(METRICS.bleu2(generated_answers, filtered_target_answers)))
    rl_scores = round5(np.mean(METRICS.rougel(generated_answers, filtered_target_answers)))
    m_scores = round5(np.mean(METRICS.meteor(generated_answers, filtered_target_answers)))
    em_scores = round5(np.mean(METRICS.exact_match(generated_answers, filtered_target_answers)))
    bs_scores = METRICS.bertscore(generated_answers, filtered_target_answers)
    none_score = round5(none_answers / len(generated_pack))

    scores = {
        'BLEU1': float(b1_scores),
        'BLEU2': float(b2_scores),
        'METEOR': float(m_scores),
        'RougeL': float(rl_scores),
        'ExactMatch': float(em_scores),
        'BertScore': bs_scores,
        'BertScore_model': BERTSCORE_MODEL_PATH,
        'NoneScore': float(none_score)
    }

    # сохраняем скоры по папку
    save_json(scores, f"{METRICS_DIR}/{pack_name}")

100%|██████████| 12/12 [22:35<00:00, 112.94s/it]


## Calculation statistics

In [3]:
metric_packs_dir = "./diaasqa/exp_logs/bfs_dmitri_with_gigachat_full(v1.3.0)/metric_packs"
metrics = [("BLEU1",), ("BLEU2",), ("METEOR",), ("RougeL",), ("ExactMatch",), ("NoneScore",), ("BertScore", "f1")]

In [4]:
packs = os.listdir(metric_packs_dir)
mean_scores = {v: [] for v in metrics}
for pack_name in packs:
    scores = json.load(open(f"{metric_packs_dir}/{pack_name}", 'r'))
    
    for m_name in metrics:
        tmp_v = scores[m_name[0]]
        if len(m_name) == 2:
            tmp_v = tmp_v[m_name[1]]

        mean_scores[m_name].append(tmp_v)

In [5]:
mean_scores = {k: np.mean(v) for k,v in mean_scores.items()}

In [6]:
mean_scores

{('BLEU1',): 0.26568666998840246,
 ('BLEU2',): 0.027993333836396534,
 ('METEOR',): 0.21943750000000004,
 ('RougeL',): 0.3502041678875685,
 ('ExactMatch',): 0.31537166666666666,
 ('NoneScore',): 0.0025,
 ('BertScore', 'f1'): 0.6517625}

In [17]:
qa_config = joblib.load("/home/dzigen/Desktop/PersonalAI/Personal-AI/experiments/qa_kg/diaasqa/exp_logs/mixture_with_gigachat_full/qa_config")
print(qa_config.query_parser_config.agent_cofig.agent_config.gen_strategy)
print(qa_config.answer_generator_config.agent_cofig.agent_config.gen_strategy)